# Investigation 2 — Spatial CN for a watershed

**Participant-directed investigation.** Follow the numbered spatial
workflow to produce a watershed boundary, land-cover–soil area table,
composite curve number, and design-runoff estimate. Verified reference
products support the complete investigation; a registered Earth Engine
project can apply the same workflow to a selected watershed.

**Minimum result:** a mapped boundary and a composite CN reported with year,
land-cover source, soil source, hydrologic condition, aggregation convention,
analysis scale, and unmapped fraction.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import json
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cnkit import composite_from_areas, runoff

# Change to "accotink_creek" for the second verified reference basin.
PREPARED_WATERSHED = "difficult_run"
ANALYSIS_YEAR = 2019
HYDROLOGIC_CONDITION = "fair"
SOILS_SOURCE = "sda"

# Set True only when the readiness check has already passed.
USE_EARTH_ENGINE = False
ADD_BACKGROUND_MAP = True

# The outlet route produces a split catchment at the supplied point.
# The gage route produces the NHDPlus aggregated upstream catchment.
DELINEATION_INPUT = "outlet"  # "outlet" or "gage"
GAGE = "01646000"
LAT, LON = 38.97594, -77.24581


## The analytical data flow

| Stage | Reference-data pathway | Earth Engine application | `cnkit` layer |
|---|---|---|---|
| Boundary | Verified workshop GeoJSON and site metadata | USGS NLDI or SS-Delineate | `cnkit.delineate` |
| Land cover | EPA StreamCat watershed percentages | Annual NLCD frequency histogram | `cnkit.gee.Basin.landcover` |
| Soils | NRCS Soil Data Access percentages | gNATSGO map-unit raster plus Soil Data Access lookup | `cnkit.gee.Basin.soil_groups` |
| Pairing | Product of marginal percentages | Pixelwise joint frequency histogram | `cnkit.gee.Basin.joint_landcover_soils` |
| CN lookup | TR-55/NEH-630 table indexed by NLCD and HSG | Same lookup table | `cnkit.lookup` |
| Composite | Area-weighted CN and area-weighted S | Same calculation | `cnkit.lookup.composite_from_areas` |

The hydrologic arithmetic is shared. Earth Engine changes how the
spatial area table is measured; it does not introduce a second
curve-number equation.


## Earth Engine data-source register

The table below records the exact Earth Engine asset identifiers used in
the guided exercise and the optional live notebook pathway. An asset
identifier documents the computational input; the agency product remains
the scientific data source that should be cited in a report.

| Workshop use | Publisher and product | Exact Earth Engine asset | Relevant band or object | Native scale | Catalog status |
|---|---|---|---|---:|---|
| Guided watershed geometry | U.S. Geological Survey, Watershed Boundary Dataset, HUC12 | `USGS/WBD/2017/HUC12` | `FeatureCollection`; field `huc12` | vector | Official Earth Engine catalog |
| Guided land cover and imperviousness | U.S. Geological Survey, NLCD 2019 release | `USGS/NLCD_RELEASES/2019_REL/NLCD` | `landcover`; `impervious` | 30 m | Official Earth Engine catalog |
| Live annual land-cover distribution and trajectory | U.S. Geological Survey, Annual NLCD Collection 1 | `projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/LANDCOVER` | first band of each annual image | 30 m | Earth Engine Community Catalog mirror |
| Live mean fractional impervious surface | U.S. Geological Survey, Annual NLCD Collection 1 | `projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/FRACTIONAL_IMPERVIOUS_SURFACE` | first band of each annual image | 30 m | Earth Engine Community Catalog mirror |
| Live soil map-unit keys for `SOILS_SOURCE="sda"` | USDA Natural Resources Conservation Service, gNATSGO | `projects/sat-io/open-datasets/gNATSGO/raster/mukey` | first band; integer map-unit key | 30 m | Earth Engine Community Catalog mirror |
| Optional global HSG comparison for `SOILS_SOURCE="hihydro"` | FutureWater, HiHydroSoil v2.0 | `projects/sat-io/open-datasets/HiHydroSoilv2_0/Hydrologic_Soil_Group_250m` | first band; modeled HSG class | 250 m | Earth Engine Community Catalog asset |

**Supporting services that are not Earth Engine assets.** The USGS NLDI
[web service](https://api.water.usgs.gov/docs/nldi/) constructs the live watershed
boundary before `cnkit` converts it to an Earth Engine geometry. For the
`sda` soil pathway, Earth Engine supplies only the gNATSGO map-unit keys;
the USDA NRCS Soil Data Access
[web service](https://sdmdataaccess.sc.egov.usda.gov/WebServiceHelp.aspx)
supplies the dominant-condition hydrologic-soil-group attributes joined to
those keys.

**Source and access documentation.** See the USGS
[Annual NLCD product page](https://www.usgs.gov/centers/eros/science/annual-national-land-cover-database),
USDA NRCS [gNATSGO documentation](https://www.nrcs.usda.gov/resources/data-and-reports/gridded-national-soil-survey-geographic-database-gnatsgo),
the Earth Engine catalog entries for
[WBD HUC12](https://developers.google.com/earth-engine/datasets/catalog/USGS_WBD_2017_HUC12)
and [NLCD 2019](https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2019_REL_NLCD),
and the [Earth Engine Community Catalog Annual NLCD record](https://gee-community-catalog.org/projects/annual_nlcd/).

The `projects/sat-io` paths are community-hosted access copies. Record the
exact asset ID and retrieval date for reproducibility, but cite the original
agency product as the data source. Availability of annual images is checked
at runtime rather than assumed from a fixed list of years.


## Step 1 — Identify the watershed and outlet

A watershed analysis begins with an outlet definition, not with a land-
cover raster. The outlet fixes the contributing area and therefore the
denominator of every percentage calculated later.

The workshop metadata distinguish the **gage coordinate**, where all
point data and Atlas 14 depths were sampled, from the **basin centroid**,
which is descriptive and should not be used as the pour point.


In [ ]:
sites = pd.read_csv(DATA_DIR / "sites.csv", dtype={"gage_number": str})
site = sites.loc[sites.watershed == PREPARED_WATERSHED].iloc[0]
boundaries = json.loads((DATA_DIR / "basins.geojson").read_text())
reference_feature = boundaries[PREPARED_WATERSHED]["features"][0]

site_summary = pd.Series(
    {
        "watershed": site["name"],
        "USGS gage": site["gage_number"],
        "published drainage area, sq mi": site["drainage_area_sqmi"],
        "gage latitude": site["sample_lat"],
        "gage longitude": site["sample_lon"],
        "basin centroid latitude": site["centroid_lat"],
        "basin centroid longitude": site["centroid_lon"],
    }
)
site_summary


## Step 2 — Delineate the watershed boundary

`cnkit` supports two scientifically distinct USGS routes.

**Outlet-coordinate route — `watershed_from_point(lat, lon)`**

1. The coordinate is validated in latitude–longitude order.
2. USGS NLDI `hydrolocation` snaps the point to an NHDPlusV2 flowline.
3. The snapped coordinate is submitted to the NLDI split-catchment
   process with upstream tracing enabled.
4. If that route is unavailable, `method="auto"` tries USGS
   SS-Delineate.
5. The returned polygon area is recomputed from its coordinates and
   compared with a minimum-area guard before a `Watershed` is returned.

**Gage route — `watershed_from_gage(gage)`**

NLDI resolves the gage identifier and returns the aggregated upstream
NHDPlusV2 catchment. It is convenient and reproducible, but it ends at a
catchment boundary rather than splitting the outlet catchment at the
exact gage coordinate. For Difficult Run this distinction is about one
NHDPlus catchment: approximately 58.15 square miles from the gage route
versus 57.82 square miles from the split-catchment route and 57.8 square
miles in the published gage metadata.

The live code is deliberately separate from Earth Engine
authentication: delineation is a USGS web-service operation and does
not use Earth Engine.


In [ ]:
live_watershed = None

if USE_EARTH_ENGINE:
    activate_full_cnkit()
    from cnkit.delineate import watershed_from_gage, watershed_from_point

    if DELINEATION_INPUT == "outlet":
        live_watershed = watershed_from_point(LAT, LON, method="auto")
    elif DELINEATION_INPUT == "gage":
        live_watershed = watershed_from_gage(GAGE)
    else:
        raise ValueError("DELINEATION_INPUT must be 'outlet' or 'gage'")

    print(live_watershed)
    print("method:       ", live_watershed.method)
    print("source:       ", live_watershed.source_dataset)
    print("area, sq mi:  ", round(live_watershed.area_sqmi, 3))
    print("request point:", live_watershed.request_point)
    print("snapped point:", live_watershed.snapped_point)
    print("warnings:     ", live_watershed.warnings)
else:
    print("Reference boundary selected:", site["name"])
    print("Set USE_EARTH_ENGINE=True to delineate the selected outlet live.")


## Step 3 — Verify and inspect the boundary

Boundary verification is an analytical step. At minimum, compare the
computed area with an independent published drainage area, inspect the
outlet position, and retain the delineation method and warnings.

The reference boundary below is a compact workshop geometry for visual
inspection. The recorded Earth Engine result also retains the full
NLDI split-catchment area and vertex count used in the live analysis.

The map requests the public **USGS Topo** basemap from The National Map.
It provides geographic names, transportation, hydrography, elevation,
land cover, and administrative context beneath the analytical boundary.
The basemap is cartographic context only; it is not used to calculate
area, land cover, soils, or curve number. Source:
[USGS National Map basemap services](https://www.usgs.gov/faqs/what-are-base-map-services-or-urls-used-national-map).


In [ ]:
recorded = None
if PREPARED_WATERSHED == "difficult_run":
    recorded = json.loads(
        (PREPARED_DIR / "difficult_run_gee_summary.json").read_text()
    )

if live_watershed is not None:
    delineated_area = live_watershed.area_sqmi
    delineation_method = live_watershed.method
elif recorded is not None:
    delineated_area = recorded["watershed"]["area_sqmi"]
    delineation_method = recorded["watershed"]["delineation"]
else:
    delineated_area = float(site["drainage_area_sqmi"])
    delineation_method = "verified workshop boundary"

area_check = pd.Series(
    {
        "published area, sq mi": float(site["drainage_area_sqmi"]),
        "delineated area, sq mi": delineated_area,
        "difference, sq mi": delineated_area - float(site["drainage_area_sqmi"]),
        "difference, percent": 100.0 * (
            delineated_area / float(site["drainage_area_sqmi"]) - 1.0
        ),
        "method": delineation_method,
    }
)
area_check


In [ ]:
import io
import urllib.parse
import urllib.request

from PIL import Image
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

def add_usgs_topo_basemap(ax, bounds):
    """Draw a USGS Topo export beneath EPSG:4326 analytical layers."""
    west, south, east, north = bounds
    parameters = {
        "bbox": ",".join(str(value) for value in bounds),
        "bboxSR": 4326,
        "imageSR": 4326,
        "size": "1000,800",
        "format": "png32",
        "transparent": "false",
        "f": "image",
    }
    endpoint = (
        "https://basemap.nationalmap.gov/arcgis/rest/services/"
        "USGSTopo/MapServer/export"
    )
    request = urllib.request.Request(
        endpoint + "?" + urllib.parse.urlencode(parameters),
        headers={"User-Agent": "cn-workshop-2026"},
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            image = Image.open(io.BytesIO(response.read())).convert("RGB")
        ax.imshow(
            np.asarray(image),
            extent=(west, east, south, north),
            interpolation="bilinear",
            zorder=0,
            aspect="auto",
        )
        return "USGS Topo loaded"
    except Exception as exc:
        ax.set_facecolor("#eef1ec")
        return "USGS Topo request noted: %s" % type(exc).__name__


geometry_for_plot = (
    live_watershed.geojson["geometry"]
    if live_watershed is not None
    else reference_feature["geometry"]
)
rings = (
    geometry_for_plot["coordinates"]
    if geometry_for_plot["type"] == "Polygon"
    else [ring for polygon in geometry_for_plot["coordinates"] for ring in polygon]
)

coordinate_arrays = [np.asarray(ring, dtype=float) for ring in rings]
all_coordinates = np.vstack(coordinate_arrays)
west, south = all_coordinates.min(axis=0)
east, north = all_coordinates.max(axis=0)
longitude_padding = max((east - west) * 0.12, 0.01)
latitude_padding = max((north - south) * 0.12, 0.01)
map_bounds = (
    west - longitude_padding,
    south - latitude_padding,
    east + longitude_padding,
    north + latitude_padding,
)

fig, ax = plt.subplots(figsize=(8.2, 6.4))
basemap_status = (
    add_usgs_topo_basemap(ax, map_bounds)
    if ADD_BACKGROUND_MAP
    else "background map disabled"
)
for coordinates in coordinate_arrays:
    ax.fill(
        coordinates[:, 0], coordinates[:, 1],
        color="#38a3a5", alpha=0.18, zorder=2,
    )
    ax.plot(
        coordinates[:, 0], coordinates[:, 1],
        color="#17274f", lw=2.0, zorder=3,
    )

marker_lat_lon = None
if live_watershed is not None:
    marker_lat_lon = live_watershed.snapped_point or live_watershed.request_point
if marker_lat_lon is None and (
    live_watershed is None or str(GAGE) == str(site["gage_number"])
):
    marker_lat_lon = (float(site["sample_lat"]), float(site["sample_lon"]))
if marker_lat_lon is not None:
    marker_lat, marker_lon = marker_lat_lon
    ax.plot(
        marker_lon, marker_lat, "o", ms=7,
        color="#b24d35", markeredgecolor="white", markeredgewidth=1.2,
        label="gage / outlet", zorder=4,
    )

map_title = (
    site["name"]
    if live_watershed is None
    else "Selected watershed — %s" % live_watershed.method
)
ax.set(
    xlim=(map_bounds[0], map_bounds[2]),
    ylim=(map_bounds[1], map_bounds[3]),
    xlabel="longitude",
    ylabel="latitude",
    title=map_title,
)
center_latitude = (map_bounds[1] + map_bounds[3]) / 2.0
ax.set_aspect(1.0 / np.cos(np.deg2rad(center_latitude)))
ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
ax.annotate(
    "N", xy=(0.965, 0.94), xytext=(0.965, 0.85),
    xycoords="axes fraction", textcoords="axes fraction",
    ha="center", va="center", fontsize=9, fontweight="bold",
    arrowprops={"arrowstyle": "-|>", "color": "#17274f", "lw": 1.4},
    zorder=5,
)
ax.text(
    0.99, 0.01, "Basemap: USGS The National Map — USGS Topo",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
    color="#17274f", bbox={"facecolor": "white", "alpha": 0.78, "edgecolor": "none"},
    zorder=5,
)
if marker_lat_lon is not None:
    ax.legend()
ax.grid(alpha=0.16, color="#54636f")
print("background map:", basemap_status)
plt.show()


## Step 4 — Initialize Earth Engine and bind the boundary

Earth Engine authentication and watershed delineation are independent.
After authentication, `cnkit.gee.Basin` converts the verified GeoJSON
geometry to an Earth Engine geometry and stores the analysis scale,
pixel limit, soil-drainage convention, request cache, and project
context. Constructing `Basin` does not yet reduce a raster.


In [ ]:
basin = None
project = None

if USE_EARTH_ENGINE:
    import os
    from getpass import getpass
    import ee
    from cnkit.gee import Basin, initialise

    project = os.environ.get("CNKIT_EE_PROJECT") or getpass(
        "Earth Engine project ID: "
    )
    ee.Authenticate()
    initialise(project=project)
    basin = Basin(live_watershed, project=project)
    print("Earth Engine basin ready at", basin.scale, "m analysis scale")
else:
    print("Reference-data pathway active; Earth Engine initialization is not used.")


## Step 5 — Measure land cover

The reference pathway reads EPA StreamCat watershed percentages. The
Earth Engine pathway calls `Basin.landcover`, which selects the requested
Annual NLCD image, clips it to the boundary, and applies one frequency-
histogram reduction. One histogram returns all classes in one request;
class-by-class masks would repeat the same spatial reduction.

The percentages include recognized classes and NoData so their
denominator remains the full analysis area. `Basin.impervious` is a
separate reduction of the fractional impervious product; developed
class percentage and impervious percentage are not interchangeable.


In [ ]:
landcover_all = pd.read_csv(DATA_DIR / "landcover_streamcat.csv")
reference_landcover = landcover_all[
    (landcover_all.watershed == PREPARED_WATERSHED)
    & (landcover_all.year == ANALYSIS_YEAR)
][["nlcd", "pct"]].copy()

live_landcover = None
live_impervious_pct = None
if basin is not None:
    live_landcover = basin.landcover(years=[ANALYSIS_YEAR])[
        ["nlcd", "pct"]
    ].copy()
    live_impervious_pct = basin.impervious(ANALYSIS_YEAR)
    print("Live Annual NLCD distribution")
    display(live_landcover.sort_values("pct", ascending=False).head(12))
    print("mean impervious surface: %.2f %%" % live_impervious_pct)
else:
    print("Reference StreamCat distribution")
    display(reference_landcover.sort_values("pct", ascending=False).head(12))

print("land-cover percentage sum:", round(
    float((live_landcover if live_landcover is not None else reference_landcover).pct.sum()), 4
))


## Step 6 — Measure hydrologic soil group

Hydrologic soil group (HSG) summarizes infiltration and transmission
behavior used by the lookup tables. The soil source is a required
argument in `cnkit`; the library does not choose one implicitly.

With `soils="sda"`, Earth Engine supplies the 30 m gNATSGO map-unit-key
raster. `cnkit` extracts only the map-unit keys present in the basin and
resolves those integers to the dominant-condition HSG through one USDA
Soil Data Access query. This avoids sending thousands of SSURGO polygons
into Earth Engine. Dual groups retain their `A/D`, `B/D`, or `C/D`
designation, and map units without an HSG remain explicitly unmapped.


In [ ]:
soils_all = pd.read_csv(DATA_DIR / "soils_hsg.csv", keep_default_na=False)
reference_soils = soils_all[
    soils_all.watershed == PREPARED_WATERSHED
][["hsg", "pct"]].copy()

live_soils = None
if basin is not None:
    live_soils = basin.soil_groups(soils=SOILS_SOURCE)
    print("Live soil distribution")
    display(live_soils)
else:
    print("Reference Soil Data Access distribution")
    display(reference_soils)

soil_table_used = live_soils if live_soils is not None else reference_soils
unmapped_soil_pct = float(
    soil_table_used.loc[
        soil_table_used.hsg.isin(["", "(none mapped)"]), "pct"
    ].sum()
)
print("soil percentage sum:       %.4f" % soil_table_used.pct.sum())
print("area with no mapped HSG:   %.2f %%" % unmapped_soil_pct)


## Step 7 — Construct land-cover–soil pairs

StreamCat supplies land-cover percentages and Soil Data Access supplies
soil-group percentages. Crossing those two marginal tables assumes they
are statistically independent:

$$
A_{ij}=A\,p(LC_i)\,p(HSG_j).
$$

The reference pathway performs that product explicitly. Earth Engine
instead observes the joint distribution by packing each pixel's land-
cover and soil codes into one integer band, applying one frequency
histogram, decoding the pairs, and converting counts to percentages.
The joint table is the principal spatial contribution: it retains which
combinations actually coexist rather than reconstructing them from two
separate summaries.


In [ ]:
def cross_marginals(lc, sg, area_column="area"):
    lc = lc.copy()
    sg = sg.copy()
    lc["key"], sg["key"] = 1, 1
    crossed = lc.merge(sg, on="key", suffixes=("_lc", "_soil"))
    crossed[area_column] = crossed.pct_lc * crossed.pct_soil / 100.0
    return crossed[["nlcd", "hsg", area_column]]

reference_pairs = cross_marginals(reference_landcover, reference_soils)
live_pairs = (
    basin.joint_landcover_soils(ANALYSIS_YEAR, soils=SOILS_SOURCE)
    if basin is not None
    else None
)

analysis_pairs = live_pairs if live_pairs is not None else reference_pairs
analysis_area_column = "pct" if live_pairs is not None else "area"

print("reference independence cross rows:", len(reference_pairs))
if live_pairs is not None:
    print("observed joint-distribution rows:", len(live_pairs))
print("analysis area sum:", round(float(analysis_pairs[analysis_area_column].sum()), 4))
display(analysis_pairs.sort_values(analysis_area_column, ascending=False).head(12))


## Step 8 — Apply the lookup to each pair

`cnkit.lookup.cn_lookup` indexes the selected hydrologic-condition row
using NLCD class and HSG. A local curve number is attached to each area
row before aggregation. Unrecognized land-cover classes and unmapped
soil groups remain `NaN`; their area is counted and reported instead of
being silently removed from the denominator.

Hydrologic condition—poor, fair, or good—is a table selection based on
cover density, residue, grazing, compaction, and related field
attributes. It is not inferred from the NLCD class itself.


In [ ]:
lookup_detail = analysis_pairs.copy()
lookup_detail["curve_number"] = cnkit.cn_lookup(
    lookup_detail.nlcd.values,
    lookup_detail.hsg.values,
    condition=HYDROLOGIC_CONDITION,
)
lookup_detail["CN_contribution"] = (
    lookup_detail[analysis_area_column] * lookup_detail.curve_number / 100.0
)
display(
    lookup_detail.sort_values("CN_contribution", ascending=False).head(15).round(4)
)
print("mapped pairs:  ", int(lookup_detail.curve_number.notna().sum()))
print("unmapped pairs:", int(lookup_detail.curve_number.isna().sum()))


## Step 9 — Aggregate and state the convention

`composite_from_areas` performs the same lookup shown above and returns
two lumped parameters:

- `cn_weighted_CN`: area-weight local CN values, following the TR-55
  Worksheet 2 procedure;
- `cn_weighted_S`: transform each local CN to retention, area-weight
  retention, and convert the result back to CN.

The function also reports the fraction of the original area table for
which no lookup was possible. It does not calculate distributed runoff;
that requires applying the runoff equation to each mapped pair before
area weighting, as demonstrated in Investigation 1.


In [ ]:
composites = {
    condition: composite_from_areas(
        analysis_pairs,
        condition=condition,
        nlcd_col="nlcd",
        hsg_col="hsg",
        area_col=analysis_area_column,
    )
    for condition in ["poor", "fair", "good"]
}

composite_table = pd.DataFrame(composites).T[
    ["cn_weighted_CN", "cn_weighted_S", "percent_area_unmapped"]
]
composite_table["poor_minus_good_CN"] = (
    composites["poor"]["cn_weighted_CN"]
    - composites["good"]["cn_weighted_CN"]
)
composite_table.round(4)


## Step 10 — Quantify the independence assumption

When a live joint table is present, the code below derives both
marginals from that same table and crosses them. Boundary, pixels, year,
soil source, and class totals are therefore held constant; the
difference isolates the independence assumption.

In the reference pathway, the recorded Difficult Run result preserves
the corresponding calculation from the executed Earth Engine analysis.


In [ ]:
if live_pairs is not None:
    lc_from_joint = live_pairs.groupby("nlcd", as_index=False)["pct"].sum()
    hsg_from_joint = live_pairs.groupby("hsg", as_index=False)["pct"].sum()
    independent_from_live = cross_marginals(
        lc_from_joint, hsg_from_joint, area_column="pct"
    )
    observed_result = composite_from_areas(
        live_pairs, condition="fair", area_col="pct"
    )
    independent_result = composite_from_areas(
        independent_from_live, condition="fair", area_col="pct"
    )
    independence_summary = {
        "marginals crossed independently": independent_result["cn_weighted_CN"],
        "observed joint distribution": observed_result["cn_weighted_CN"],
        "independence assumption, CN": (
            independent_result["cn_weighted_CN"]
            - observed_result["cn_weighted_CN"]
        ),
    }
    print(pd.Series(independence_summary).round(4))
elif recorded is not None:
    comparison = recorded["curve_number_2019_fair"]
    print("same live raster marginals crossed independently : %.4f" % comparison["marginals_crossed_independently"])
    print("live observed joint distribution                 : %.4f" % comparison["observed_joint_distribution"])
    print("independence assumption                          : %+.4f CN" % comparison["independence_assumption_cn"])
    print("raster soil area with no HSG                     : %.2f %%" % recorded["soils"]["percent_area_no_hsg"])
else:
    print("Reference marginal analysis complete for", site["name"])


For the recorded 2019 Difficult Run analysis, the raster marginals give
CN 77.6638 when crossed independently and CN 75.7790 when their observed
pixelwise joint distribution is used. The 1.8848-unit difference is an
empirical estimate of this assumption for that boundary, year, soil
source, and analysis scale.


## Step 11 — Calculate design runoff and assemble provenance

The composite CN is a parameter; a runoff depth additionally requires a
storm depth and lambda. The ten-year, twenty-four-hour rainfall below is
taken from the versioned NOAA Atlas 14 table at the reference gage.

The provenance object places data source, year, boundary, soil source,
condition, compositing convention, unmapped area, and rainfall basis
beside the numerical result. A reviewer should not have to reconstruct
those choices from the code.


In [ ]:
atlas14 = pd.read_csv(DATA_DIR / "atlas14_depths.csv")
design_depth = float(
    atlas14.loc[
        (atlas14.watershed == PREPARED_WATERSHED)
        & (atlas14.duration == "24-hr")
        & (atlas14.ari_years == 10),
        "depth_in",
    ].iloc[0]
)
headline = composites[HYDROLOGIC_CONDITION]
design_runoff = float(
    runoff(design_depth, headline["cn_weighted_CN"], lam=0.20)
)

provenance = {
    "watershed": site["name"],
    "boundary_method": delineation_method,
    "boundary_area_sqmi": round(float(delineated_area), 4),
    "analysis_path": "Earth Engine observed joint" if live_pairs is not None else "reference marginal cross",
    "land_cover": "Annual NLCD" if live_pairs is not None else "EPA StreamCat NLCD summary",
    "land_cover_year": ANALYSIS_YEAR,
    "soils": SOILS_SOURCE if live_pairs is not None else "NRCS Soil Data Access summary",
    "hydrologic_condition": HYDROLOGIC_CONDITION,
    "composite_convention": "area-weighted curve number",
    "unmapped_area_pct": round(float(headline["percent_area_unmapped"]), 4),
    "lambda": 0.20,
    "design_storm": "NOAA Atlas 14, 10-year 24-hour at reference gage",
    "design_depth_in": design_depth,
    "composite_cn": round(float(headline["cn_weighted_CN"]), 4),
    "runoff_depth_in": round(design_runoff, 4),
}
print(json.dumps(provenance, indent=2))


## What the convenience method does

For a live `Basin`, the production call

```python
basin.composite_cn(year, condition="fair", soils="sda")
```

performs Steps 7 through 9 by calling
`joint_landcover_soils` and then delegating the hydrologic arithmetic to
`cnkit.lookup.composite_from_areas`. It returns the composite results
together with asset identifiers, scale, soil source, boundary area,
pair count, and unmapped fractions. The notebook used the lower-level
calls so that the intermediate area table and every assumption remain
available for inspection.


## Method audit — Library operation and analyst decision

| Step | Call or object | Operation performed by `cnkit` | Decision or check retained by the analyst |
|---|---|---|---|
| 1–3 | `watershed_from_point`, `watershed_from_gage`, `Watershed` | Resolve the outlet, obtain upstream geometry, calculate area, and retain method provenance | Select the outlet meaning and verify boundary, area, and warnings |
| 4 | `Basin` | Convert the checked boundary to an Earth Engine geometry and retain scale, project, pixel, drainage, and cache settings | Select project, scale, and drainage convention |
| 5 | `Basin.landcover`, `Basin.impervious` | Reduce categorical Annual NLCD and fractional imperviousness separately | Select year and interpret class area separately from impervious fraction |
| 6 | `Basin.soil_groups` | Summarize map-unit keys and resolve the occurring keys through Soil Data Access | Select soil source and account for dual or unmapped groups |
| 7 | `Basin.joint_landcover_soils` | Pack two raster codes, execute one histogram, and decode observed pairs | Decide whether joint spatial evidence or a marginal approximation supports the analysis |
| 8 | `cn_lookup` | Index the crosswalk by NLCD, HSG, and hydrologic condition | Establish the crosswalk and condition basis |
| 9 | `composite_from_areas` | Calculate weighted-CN and weighted-retention composites and unmapped fractions | Select and report the aggregation convention |
| 10 | marginal cross and joint comparison | Recalculate from controlled marginals to isolate the independence effect | Interpret the difference at the stated boundary, year, source, and scale |
| 11 | `runoff` | Apply the event equation to the selected storm, CN, and lambda | Select the rainfall basis and retain complete provenance |


## Open investigation — Choose a question

1. Compare the observed joint land-cover–soil distribution with the product
   of its marginal distributions.
2. Change the watershed, analysis year, hydrologic condition, or soil source
   while holding the other inputs fixed.
3. Examine whether unmapped area or boundary choice materially affects the
   reported composite.
4. Compare the effect on CN with the effect on runoff for the stated design
   rainfall depth.

Change one analytical choice at a time and preserve the baseline result.


## Reporting record

Record: watershed and outlet; boundary method and area; land-cover and soil
sources; year and scale; lookup and aggregation conventions; composite CN;
unmapped fraction; design runoff; extension result; and one limitation.

## References and data sources

- U.S. Geological Survey. *Annual National Land Cover Database*.
  [Collection 1 products and citation](https://www.usgs.gov/centers/eros/science/annual-national-land-cover-database).
- USDA Natural Resources Conservation Service.
  [gNATSGO](https://www.nrcs.usda.gov/resources/data-and-reports/gridded-national-soil-survey-geographic-database-gnatsgo)
  and [Soil Data Access](https://sdmdataaccess.sc.egov.usda.gov/WebServiceHelp.aspx).
- U.S. Environmental Protection Agency. [StreamCat Dataset](https://www.epa.gov/national-aquatic-resource-surveys/streamcat-dataset).
- U.S. Geological Survey. [Network Linked Data Index documentation](https://api.water.usgs.gov/docs/nldi/).
- NOAA National Weather Service. [Atlas 14](https://www.weather.gov/owp/hdsc).
- USDA NRCS. 2004. [NEH Part 630, Chapter 10](https://directives.nrcs.usda.gov/sites/default/files2/1712930608/7300.pdf).

The complete cross-notebook source ledger is available in
[workshop source ledger](https://github.com/skp703/cn-workshop-2026/blob/main/docs/SOURCES.md).
